# Optional: retention (MMLU / ARC-Challenge) on verified full exports

**Not part of the 15-run core campaign and never launched automatically.** Run it only as an explicit, separately planned follow-up.

What changed versus the submitted version of this notebook:
* It evaluates only **verified full inference exports** (`export/export_metadata.json` with `status: ok`). The submitted runs saved PEFT adapter-only checkpoints, which **drop AutoMode's base-mode updates**; evaluating those as "AutoMode" was invalid, so they are refused here.
* `MAX_EVAL_SAMPLES` is now actually passed to MMLU (`max_samples=`); before, MMLU silently ran all 14k items.
* MMLU/ARC use their own context budget (`mc_max_source_len`, default 2048), never truncate the test question, drop few-shot exemplars only when needed and log the shots actually used per item.
* The MMLU subset is a fixed shuffle (`MC_SUBSET_SEED`), identical for every model and seed. ARC uses its first `MAX_EVAL_SAMPLES` items.
* Failures raise; nothing is recorded as a result unless both evaluations finished.

These numbers are **not comparable** with retention numbers from the old parser/protocol.

In [ ]:
import json
from pathlib import Path
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from automode.config import RunConfig
from automode.eval import evaluate_mmlu, evaluate_arc, MC_SUBSET_SEED
from automode.train import save_json

RESULTS_ROOTS = [Path('PATH/TO/worker0/results')]   # rebuttal output dirs that contain core/<job>/...
OUT_ROOT = Path('retention_eval_v1')                # fresh directory for these results
MAX_EVAL_SAMPLES = 2000                              # applied to MMLU AND ARC; None = full test sets
DEVICE = 'cuda:0'
OUT_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
# ── Find verified full exports; refuse adapter-only checkpoints ─────────────
exports = []
for root in RESULTS_ROOTS:
    for meta_path in sorted(root.glob('core/*/attempts/*/export/export_metadata.json')):
        meta = json.loads(meta_path.read_text())
        export_dir = meta_path.parent
        attempt = export_dir.parent
        if meta.get('status') != 'ok' or not list(export_dir.glob('*.safetensors')):
            print(f'[skip: unverified or weights missing] {export_dir}')
            continue
        exports.append((export_dir, attempt, meta))
    for adapter_cfg in root.rglob('adapter_config.json'):
        print(f'[refused: adapter-only checkpoint, loses AutoMode base updates] {adapter_cfg.parent}')
print(f'{len(exports)} verified full exports')
for export_dir, attempt, meta in exports:
    print(' ', meta['job_id'], meta['identity'][:12], export_dir)

In [ ]:
# ── Evaluate each export; one result file per export identity ──────────────
rows = []
for export_dir, attempt, meta in exports:
    out_dir = OUT_ROOT / meta['identity']
    done_path = out_dir / 'retention.json'
    if done_path.exists() and json.loads(done_path.read_text()).get('identity') == meta['identity']:
        rows.append(json.loads(done_path.read_text()))
        print(f"[done] {meta['job_id']}")
        continue
    cfg = RunConfig(**json.loads((attempt / 'configs' / 'run_config.json').read_text()))
    cfg.eval_benchmarks = ('mmlu', 'arc_c')
    cfg.max_eval_samples = MAX_EVAL_SAMPLES
    cfg.device = DEVICE
    paths = {'evals': out_dir / 'evals'}
    paths['evals'].mkdir(parents=True, exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained(export_dir)
    model = AutoModelForCausalLM.from_pretrained(export_dir, torch_dtype=torch.bfloat16,
                                                 device_map={'': DEVICE},
                                                 attn_implementation=cfg.attn_implementation)
    mmlu = evaluate_mmlu(model, tokenizer, cfg, paths, max_samples=cfg.max_eval_samples)
    arc = evaluate_arc(model, tokenizer, cfg, paths)
    row = {'identity': meta['identity'], 'job_id': meta['job_id'], 'method': cfg.method, 'seed': cfg.seed,
           'variant': cfg.variant_label(), 'protocol_version': cfg.protocol_version,
           'export_dir': str(export_dir), 'max_eval_samples': MAX_EVAL_SAMPLES,
           'mmlu_subset_seed': MC_SUBSET_SEED, 'mc_max_source_len': cfg.mc_max_source_len, **mmlu, **arc}
    save_json(row, done_path)
    rows.append(row)
    print(row)
    del model
    torch.cuda.empty_cache()

import pandas as pd
pd.DataFrame(rows)